In [ ]:
# Problema: Evaluar si un extracto tributario real puede usarse para análisis por código postal.
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'submission').is_dir())
SOURCE, OUTPUT = ROOT / 'data/vemont.csv', ROOT / 'submission/quality_report.csv'


In [ ]:
# La clave analítica es código postal y tramo de ingreso; zipcode 0 es el total estatal.
tax = pd.read_csv(SOURCE)
tax.shape, tax[['zipcode', 'agi_stub', 'N1', 'A00100']].head()


In [ ]:
checks = [('required_columns', 'completeness', len({'STATE', 'zipcode', 'agi_stub', 'N1', 'A00100'} - set(tax.columns))), ('income_group_domain', 'validity', int((~tax.agi_stub.between(1, 6)).sum())), ('postal_income_key_unique', 'uniqueness', int(tax[['zipcode', 'agi_stub']].duplicated().sum())), ('return_count_nonnegative', 'validity', int((tax.N1 < 0).sum())), ('statewide_total_separated', 'scope', int(tax.zipcode.eq(0).sum()))]
report = pd.DataFrame(checks, columns=['rule_name', 'dimension', 'violations'])
report['status'] = ['FAIL' if name == 'statewide_total_separated' and count else 'PASS' for name, _, count in checks]
report.to_csv(OUTPUT, index=False)
report
